# 05_b — Indicadores socioeconômicos dos municípios do RN (2004–2025)

**Objetivo:** coletar, para os mesmos anos da série de repasses (05_a), os indicadores que permitem comparar o
dinheiro recebido com o tamanho e a riqueza de cada município.

| Indicador | Fonte | Anos com dado | Observação |
|---|---|---|---|
| População | IBGE / SIDRA | 2004–2025 | Estimativas anuais + Contagem 2007 + Censos 2010 e 2022; **2023 estimado** |
| PIB municipal | IBGE / SIDRA (tabela 5938) | 2004–2023 | **2024 e 2025 ainda não publicados** |
| IPCA | IBGE / SIDRA (tabela 1737) | 2004–2025 | Usado para corrigir valores pela inflação |
| CadÚnico | MDS / VIS DATA | quando a API responder | API **fora do ar desde 21/10/2026** (manutenção do MDS) |

**Decisões do grupo:**
- período de **2004 a 2025**, igual aos repasses; os anos sem PIB ficam vazios e marcados;
- **população de cada ano** (e não fixa em 2022 como no notebook 01): em 20 anos os municípios mudaram muito;
- **CadÚnico preparado:** o código tenta a API e, se ela não responder, usa a cópia do GitHub (dezembro de 2020
  a 2025, baixada antes da manutenção). Na série longa, a medida de pobreza principal passa a ser o PIB per capita.

**Sequência:** 05_a (repasses) → **05_b (indicadores)** → 05_c (junção, análises e gráficos).

**Etapas do notebook**
1. Bibliotecas, parâmetros e plano B
2. População
3. PIB municipal
4. IPCA
5. CadÚnico
6. Junção e conferência
7. Saída `indicadores_2004_2025.csv`

## 1. Bibliotecas, parâmetros e plano B

Como no 05_a, cada fonte externa tem um **plano B**: se a API não responder (o Colab roda fora do Brasil e os
sites do governo às vezes recusam essas conexões), o notebook lê uma cópia do mesmo dado guardada no repositório,
na pasta `dados/`. O notebook avisa qual das duas foi usada.

In [1]:
import os
import numpy as np
import pandas as pd
import requests

ANO_INICIO, ANO_FIM = 2004, 2025
ANOS = list(range(ANO_INICIO, ANO_FIM + 1))
UF_IBGE = 24                                   # código IBGE do RN

REPO = 'Renata-Nascimento/Projeto-U02-CD'
RAW_GITHUB = f'https://raw.githubusercontent.com/{REPO}/main/'
os.makedirs('saida', exist_ok=True)
os.makedirs('copias_para_github', exist_ok=True)   # cópias atualizadas das APIs, para subir em dados/ quando mudarem

def com_plano_b(nome, buscar_na_api, copia_no_github, colunas_se_faltar=None):
    """Tenta buscar_na_api(); se der erro, lê a cópia do GitHub. Quando a API funciona, salva uma cópia nova em
    copias_para_github/ (com o mesmo nome de arquivo usado no repositório).
    Para indicadores complementares (colunas_se_faltar preenchido), se nem a cópia existir, segue com tabela vazia."""
    try:
        tabela = buscar_na_api()
        tabela.to_csv(f"copias_para_github/{copia_no_github.split('/')[-1]}", index=False)
        print(f'{nome}: API')
        return tabela
    except Exception as erro:
        print(f'{nome}: API não respondeu ({erro.__class__.__name__}); usando a cópia do GitHub')
    try:
        return pd.read_csv(RAW_GITHUB + copia_no_github)
    except Exception:
        if colunas_se_faltar is None:
            raise
        print(f'{nome}: cópia do GitHub também indisponível; seguindo sem esse indicador')
        return pd.DataFrame(columns=colunas_se_faltar)

def sidra(caminho):
    """Consulta a API do SIDRA (IBGE) e devolve uma tabela (a 1ª linha da resposta é o cabeçalho)."""
    resposta = requests.get('https://apisidra.ibge.gov.br/values' + caminho, timeout=(20, 120))
    resposta.raise_for_status()
    return pd.DataFrame(resposta.json()[1:])

In [2]:
municipios = com_plano_b(
    'Lista de municípios',
    lambda: pd.DataFrame([{'cod_ibge': m['id'], 'municipio': m['nome']} for m in requests.get(
        f'https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_IBGE}/municipios', timeout=20).json()]),
    'dados/ibge/municipios_rn.csv')[['cod_ibge', 'municipio']]
print(len(municipios), 'municípios')

Lista de municípios: API não respondeu (ConnectTimeout); usando a cópia do GitHub
167 municípios


## 2. População

O IBGE não tem uma tabela única de população para todos os anos. Juntamos quatro:

| Anos | Tabela SIDRA | O que é |
|---|---|---|
| 2004–2006, 2008–2009, 2011–2021, 2024–2025 | 6579 | Estimativa anual (usada pelo TCU para repartir o FPM) |
| 2007 | 793 | Contagem da População 2007 |
| 2010 | 1378 | Censo 2010 |
| 2022 | 4709 | Censo 2022 |
| **2023** | — | **Não publicado:** estimamos pela média de 2022 e 2024 |

**Atenção aos "degraus" da série** (aparecem na tabela abaixo, na coluna `variação_%`):
- **2022 (−7,3%):** o Censo contou cerca de 7% menos habitantes do que a estimativa de 2021 indicava;
- **2013 (+4,5%) e 2018 (−0,8%):** o IBGE revisou o método das estimativas nesses anos;
- **2007:** a Contagem ficou um pouco abaixo das estimativas vizinhas.

Esses saltos vêm da **troca ou revisão da fonte**, não de mudanças reais de população de um ano para o outro.
Por isso a coluna `fonte_populacao` fica na base, e nas análises de valores per capita esses anos devem ser lidos
com cuidado.

In [3]:
def buscar_populacao():
    anos_estimativa = ','.join(str(a) for a in ANOS if a not in (2007, 2010, 2022, 2023))
    consultas = [
        (f'/t/6579/n6/in%20n3%20{UF_IBGE}/v/9324/p/{anos_estimativa}', 'estimativa'),
        (f'/t/793/n6/in%20n3%20{UF_IBGE}/v/93/p/2007', 'contagem 2007'),
        (f'/t/1378/n6/in%20n3%20{UF_IBGE}/v/93/p/2010/c1/0/c2/0/c287/0', 'censo 2010'),
        (f'/t/4709/n6/in%20n3%20{UF_IBGE}/v/93/p/2022', 'censo 2022'),
    ]
    partes = []
    for caminho, fonte in consultas:
        t = sidra(caminho)[['D1C', 'D3N', 'V']]
        t.columns = ['cod_ibge', 'ano', 'populacao']
        t['fonte_populacao'] = fonte
        partes.append(t)
    return pd.concat(partes, ignore_index=True)

populacao = com_plano_b('População', buscar_populacao, 'dados/ibge/populacao_rn.csv')
populacao = populacao.astype({'cod_ibge': int, 'ano': int})
populacao['populacao'] = pd.to_numeric(populacao['populacao'], errors='coerce')

# 2023: média de 2022 (Censo) e 2024 (estimativa)
pop_2023 = (populacao[populacao['ano'].isin([2022, 2024])]
            .groupby('cod_ibge', as_index=False)['populacao'].mean().round())
pop_2023['ano'] = 2023
pop_2023['fonte_populacao'] = 'imputada (média 2022 e 2024)'
populacao = (pd.concat([populacao, pop_2023], ignore_index=True)
             .query('@ANO_INICIO <= ano <= @ANO_FIM')
             .sort_values(['cod_ibge', 'ano']).reset_index(drop=True))
populacao['populacao'] = populacao['populacao'].astype('Int64')

População: API


In [4]:
# Total do RN por ano e fonte (os "degraus" de 2010 e 2022 são os Censos corrigindo as estimativas)
total_pop = populacao.groupby(['ano', 'fonte_populacao'], as_index=False).agg(
    municipios=('cod_ibge', 'nunique'), populacao_RN=('populacao', 'sum'))
total_pop['variação_%'] = total_pop['populacao_RN'].pct_change() * 100
total_pop.round(1)

,ano,fonte_populacao,municipios,populacao_RN,variação_%
0,2004,estimativa,167,2962107,<NA>
1,2005,estimativa,167,3003087,1.4
2,2006,estimativa,167,3043760,1.4
3,2007,contagem 2007,167,3013740,-1.0
4,2008,estimativa,167,3106436,3.1
5,2009,estimativa,167,3137541,1.0
6,2010,censo 2010,167,3168027,1.0
7,2011,estimativa,167,3198657,1.0
8,2012,estimativa,167,3228198,0.9
9,2013,estimativa,167,3373959,4.5


## 3. PIB municipal

Tabela 5938 do SIDRA, PIB a preços correntes, em **milhares de reais** (R$ mil): um valor de 163.709 significa R$ 163,7 milhões. O IBGE publica com cerca de 2 anos de atraso, então **2024 e 2025 ficam vazios**.

In [5]:
def buscar_pib():
    t = sidra(f'/t/5938/n6/in%20n3%20{UF_IBGE}/v/37/p/' + ','.join(str(a) for a in ANOS if a <= 2023))
    t = t[['D1C', 'D3N', 'V']]
    t.columns = ['cod_ibge', 'ano', 'pib_mil_reais']
    return t

pib = com_plano_b('PIB', buscar_pib, 'dados/ibge/pib_rn.csv')
pib = pib.astype({'cod_ibge': int, 'ano': int})
pib['pib_mil_reais'] = pd.to_numeric(pib['pib_mil_reais'], errors='coerce')

(pib.groupby('ano').agg(municipios=('cod_ibge', 'nunique'), pib_RN_R_bilhoes=('pib_mil_reais', 'sum'))
 .assign(pib_RN_R_bilhoes=lambda d: d['pib_RN_R_bilhoes'] / 1e6).round(2))

PIB: API


,municipios,pib_RN_R_bilhoes
ano,,
2004,167,17.25
2005,167,19.97
2006,167,22.89
2007,167,26.32
2008,167,28.90
2009,167,30.94
2010,167,36.18
2011,167,40.99
2012,167,46.41


## 4. IPCA

Tabela 1737 do SIDRA, número-índice mensal. Calculamos a média de cada ano e um **fator** que leva valores de
cada ano para **reais de 2025** (último ano da série). Exemplo: R$ 1 de 2004 vale cerca de R$ 3 em 2025.

Os valores reais (repasses e PIB corrigidos) serão calculados no 05_c: aqui só preparamos o fator.

In [6]:
def buscar_ipca():
    t = sidra(f'/t/1737/n1/all/v/2266/p/{ANO_INICIO}01-{ANO_FIM}12')[['D3C', 'V']]
    t.columns = ['anomes', 'ipca_indice']
    return t

ipca_mensal = com_plano_b('IPCA', buscar_ipca, 'dados/ibge/ipca_mensal.csv')
ipca_mensal['ano'] = ipca_mensal['anomes'].astype(str).str[:4].astype(int)
ipca_mensal['ipca_indice'] = pd.to_numeric(ipca_mensal['ipca_indice'], errors='coerce')

ipca = ipca_mensal.groupby('ano', as_index=False).agg(meses=('ipca_indice', 'count'),
                                                      ipca_indice_medio=('ipca_indice', 'mean'))
ipca[f'fator_ipca_{ANO_FIM}'] = ipca.loc[ipca['ano'] == ANO_FIM, 'ipca_indice_medio'].iloc[0] / ipca['ipca_indice_medio']
ipca.round(3)

IPCA: API


,ano,meses,ipca_indice_medio,fator_ipca_2025
0,2004,12,2318.145,3.149
1,2005,12,2477.391,2.947
2,2006,12,2581.034,2.829
3,2007,12,2675.017,2.729
4,2008,12,2826.920,2.583
5,2009,12,2965.101,2.462
6,2010,12,3114.504,2.344
7,2011,12,3321.197,2.198
8,2012,12,3500.658,2.086
9,2013,12,3717.849,1.964


## 5. CadÚnico

API pública do MDS (VIS DATA), posição de **dezembro** de cada ano. O código IBGE do MDS tem **6 dígitos**
(sem o dígito verificador).

**Situação em outubro de 2026:** a API está fora do ar desde 21/10 ("manutenção preventiva de segurança", segundo
o aviso do MDS). Nesse caso o notebook usa a cópia do GitHub, que tem **dezembro de 2020 a 2025** (baixada antes
da manutenção). Quando a API voltar, é só rodar de novo: ela traz todos os anos que o MDS tiver.

In [7]:
CAMPOS_CADUNICO = {
    'codigo_ibge': 'cod_ibge6',
    'anomes_s': 'anomes',
    'cadun_qtd_familias_cadastradas_i': 'cad_familias',
    'cadun_qtd_pessoas_cadastradas_i': 'cad_pessoas',
    'cadun_qtd_pessoas_cadastradas_rfpc_ate_meio_sm_i': 'cad_baixa_renda',
}

def buscar_cadunico():
    params = {
        'q': '*:*',
        'fq': ['sigla_uf:RN', 'anomes_s:(' + ' OR '.join(f'{a}12' for a in ANOS) + ')'],
        'fl': ','.join(CAMPOS_CADUNICO), 'rows': 10000, 'wt': 'json',
    }
    r = requests.get('https://aplicacoes.mds.gov.br/sagi/servicos/misocial', params=params, timeout=(20, 120))
    r.raise_for_status()
    t = pd.DataFrame(r.json()['response']['docs']).rename(columns=CAMPOS_CADUNICO)   # erro se vier página de aviso
    t['ano'] = t['anomes'].astype(str).str[:4].astype(int)
    return t.drop(columns='anomes')

cadunico = com_plano_b('CadÚnico', buscar_cadunico, 'dados/cadunico/cadunico_dezembro_rn.csv',
                       colunas_se_faltar=['cod_ibge6', 'ano', 'cad_familias', 'cad_pessoas', 'cad_baixa_renda'])
cadunico = cadunico.astype({'cod_ibge6': int, 'ano': int})
for coluna in ['cad_familias', 'cad_pessoas', 'cad_baixa_renda']:
    cadunico[coluna] = pd.to_numeric(cadunico[coluna], errors='coerce')

(cadunico.groupby('ano').agg(municipios=('cod_ibge6', 'nunique'),
                             municipios_com_dado=('cad_pessoas', 'count'),
                             pessoas_cadastradas=('cad_pessoas', 'sum')))

CadÚnico: API não respondeu (JSONDecodeError); usando a cópia do GitHub


,municipios,municipios_com_dado,pessoas_cadastradas
ano,,,
2020,167,167,1717404
2021,167,167,1830511
2022,167,167,2027757
2023,167,167,2078471
2024,167,167,2006894
2025,167,167,2012453


## 6. Junção e conferência

Grade completa: **167 municípios × 22 anos = 3.674 linhas**. Cada indicador entra com *left join*, então os anos
sem dado aparecem como vazio (e não somem).

In [8]:
indicadores = (municipios
               .merge(pd.DataFrame({'ano': ANOS}), how='cross')
               .merge(populacao, on=['cod_ibge', 'ano'], how='left')
               .merge(pib, on=['cod_ibge', 'ano'], how='left')
               .merge(ipca[['ano', 'ipca_indice_medio', f'fator_ipca_{ANO_FIM}']], on='ano', how='left')
               .assign(cod_ibge6=lambda d: d['cod_ibge'] // 10)
               .merge(cadunico, on=['cod_ibge6', 'ano'], how='left')
               .drop(columns='cod_ibge6')
               .sort_values(['cod_ibge', 'ano']).reset_index(drop=True))
print(indicadores.shape)
indicadores.head()

(3674, 11)


,cod_ibge,municipio,ano,populacao,fonte_populacao,pib_mil_reais,ipca_indice_medio,fator_ipca_2025,cad_familias,cad_pessoas,cad_baixa_renda
0,2400109,Acari,2004,11282,estimativa,31036.0,2318.145000,3.149433,NaN,NaN,NaN
1,2400109,Acari,2005,11303,estimativa,35205.0,2477.390833,2.946988,NaN,NaN,NaN
2,2400109,Acari,2006,11324,estimativa,39731.0,2581.034167,2.828650,NaN,NaN,NaN
3,2400109,Acari,2007,10911,contagem 2007,43491.0,2675.016667,2.729270,NaN,NaN,NaN
4,2400109,Acari,2008,11222,estimativa,48200.0,2826.920000,2.582613,NaN,NaN,NaN


In [9]:
# Cobertura: quantos municípios têm cada indicador em cada ano (o máximo é 167)
cobertura = indicadores.groupby('ano')[['populacao', 'pib_mil_reais', 'ipca_indice_medio', 'cad_pessoas']].count()
cobertura

,populacao,pib_mil_reais,ipca_indice_medio,cad_pessoas
ano,,,,
2004,167,167,167,0
2005,167,167,167,0
2006,167,167,167,0
2007,167,167,167,0
2008,167,167,167,0
2009,167,167,167,0
2010,167,167,167,0
2011,167,167,167,0
2012,167,167,167,0


In [10]:
# Checagens automáticas
assert len(indicadores) == 167 * len(ANOS), 'grade incompleta'
assert not indicadores.duplicated(['cod_ibge', 'ano']).any(), 'município-ano repetido'
assert indicadores['populacao'].notna().all(), 'falta população em algum município-ano'
assert indicadores['ipca_indice_medio'].notna().all(), 'falta IPCA em algum ano'
anos_sem_pib = sorted(indicadores.loc[indicadores['pib_mil_reais'].isna(), 'ano'].unique())
assert anos_sem_pib == [2024, 2025], f'PIB faltando em anos inesperados: {anos_sem_pib}'
print('OK: 3.674 linhas, população e IPCA completos, PIB vazio só em 2024 e 2025 (ainda não publicado)')
print('CadÚnico disponível em:', [int(a) for a in sorted(indicadores.loc[indicadores['cad_pessoas'].notna(), 'ano'].unique())])

OK: 3.674 linhas, população e IPCA completos, PIB vazio só em 2024 e 2025 (ainda não publicado)
CadÚnico disponível em: [2020, 2021, 2022, 2023, 2024, 2025]


## 7. Saída

In [11]:
nome_saida = f'saida/indicadores_{ANO_INICIO}_{ANO_FIM}.csv'
indicadores.to_csv(nome_saida, index=False)

dicionario_indicadores = pd.DataFrame([
    ['cod_ibge', 'Código IBGE do município (chave de integração)', 'IBGE', '-'],
    ['municipio', 'Nome oficial do município', 'IBGE', '-'],
    ['ano', f'Ano de referência ({ANO_INICIO} a {ANO_FIM})', '-', 'ano'],
    ['populacao', 'População residente no ano', 'IBGE (SIDRA 6579, 793, 1378, 4709)', 'pessoas'],
    ['fonte_populacao', 'Origem da população: estimativa, contagem 2007, censo 2010, censo 2022 ou imputada (2023)',
     'Derivada', 'categoria'],
    ['pib_mil_reais', 'PIB a preços correntes (vazio em 2024 e 2025, ainda não publicados)', 'IBGE (SIDRA 5938)',
     'R$ mil nominais'],
    ['ipca_indice_medio', 'Média anual do número-índice do IPCA', 'IBGE (SIDRA 1737)', 'índice'],
    [f'fator_ipca_{ANO_FIM}', f'Multiplica um valor do ano para convertê-lo em reais de {ANO_FIM}', 'Derivada', 'razão'],
    ['cad_familias', 'Famílias inscritas no CadÚnico em dezembro', 'MDS (VIS DATA)', 'famílias'],
    ['cad_pessoas', 'Pessoas inscritas no CadÚnico em dezembro', 'MDS (VIS DATA)', 'pessoas'],
    ['cad_baixa_renda', 'Pessoas no CadÚnico com renda até ½ salário mínimo per capita, em dezembro', 'MDS (VIS DATA)',
     'pessoas'],
], columns=['variável', 'descrição', 'fonte', 'unidade'])
dicionario_indicadores['tipo_dado'] = dicionario_indicadores['variável'].map(indicadores.dtypes.astype(str))
dicionario_indicadores.to_csv(f'saida/dicionario_indicadores_{ANO_INICIO}_{ANO_FIM}.csv', index=False)

print('Arquivos salvos:', nome_saida, f'e saida/dicionario_indicadores_{ANO_INICIO}_{ANO_FIM}.csv')
dicionario_indicadores

Arquivos salvos: saida/indicadores_2004_2025.csv e saida/dicionario_indicadores_2004_2025.csv


,variável,descrição,fonte,unidade,tipo_dado
0,cod_ibge,Código IBGE do município (chave de integração),IBGE,-,int64
1,municipio,Nome oficial do município,IBGE,-,object
2,ano,Ano de referência (2004 a 2025),-,ano,int64
3,populacao,População residente no ano,"IBGE (SIDRA 6579, 793, 1378, 4709)",pessoas,Int64
4,fonte_populacao,"Origem da população: estimativa, contagem 2007...",Derivada,categoria,object
5,pib_mil_reais,"PIB a preços correntes (vazio em 2024 e 2025, ...",IBGE (SIDRA 5938),R$ mil nominais,float64
6,ipca_indice_medio,Média anual do número-índice do IPCA,IBGE (SIDRA 1737),índice,float64
7,fator_ipca_2025,Multiplica um valor do ano para convertê-lo em...,Derivada,razão,float64
8,cad_familias,Famílias inscritas no CadÚnico em dezembro,MDS (VIS DATA),famílias,float64
9,cad_pessoas,Pessoas inscritas no CadÚnico em dezembro,MDS (VIS DATA),pessoas,float64


## Resumo do notebook

- **Base de indicadores 2004–2025:** 167 municípios × 22 anos, com população, PIB, IPCA e CadÚnico.
- **População de cada ano**, juntando estimativas, Contagem 2007 e Censos 2010 e 2022; 2023 estimado.
  A coluna `fonte_populacao` mostra a origem, porque os Censos criam "degraus" na série.
- **PIB** vazio em 2024 e 2025 (ainda não publicado pelo IBGE).
- **CadÚnico** só nos anos disponíveis (hoje, cópia de 2020 a 2025, porque a API do MDS está em manutenção).
- **Plano B** em todas as fontes: se a API falhar, o notebook usa a cópia guardada em `dados/` no GitHub.